## EDA
Explore the three bronze tables to identify data quality issues, missingness patterns, and categorical inconsistencies and to define the concrete transformation rules for the silver layer.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import pandas as pd
from pyspark.sql import functions as F

## 1. Facilities

In [0]:
%sql
-- Schema and types
DESCRIBE nem_project.1_bronze.facilities;

In [0]:
%sql
-- Sample rows
select * from nem_project.`1_bronze`.facilities limit 20

In [0]:
%sql
-- Row count and basic shape
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT facility_code) AS distinct_facilities,
       COUNT(DISTINCT unit_code) AS distinct_units,
       COUNT(DISTINCT network_region) AS distinct_regions
FROM nem_project.1_bronze.facilities;

In [0]:
%sql
-- Count Null and missingness records per column.
SELECT
  SUM(CASE WHEN lat IS NULL OR lat = '' THEN 1 ELSE 0 END) AS missing_lat,
  SUM(CASE WHEN lng IS NULL OR lng = '' THEN 1 ELSE 0 END) AS missing_lng,
  SUM(CASE WHEN fueltech_id IS NULL OR fueltech_id = '' THEN 1 ELSE 0 END) AS missing_fueltech,
  SUM(CASE WHEN status_id IS NULL OR status_id = '' THEN 1 ELSE 0 END) AS missing_status,
  SUM(CASE WHEN capacity_registered IS NULL OR capacity_registered = '' THEN 1 ELSE 0 END) AS missing_capacity_registered
FROM nem_project.1_bronze.facilities;

In [0]:
%sql
-- Check duplicate records
SELECT facility_code, unit_code, COUNT(*) AS n
FROM nem_project.1_bronze.facilities
GROUP BY facility_code, unit_code
HAVING COUNT(*) > 1
LIMIT 20;

#### Further revision for each column

In [0]:
%sql
-- Network id
SELECT DISTINCT network_id FROM nem_project.1_bronze.facilities ORDER BY network_id;

In [0]:
%sql
-- Network region
SELECT DISTINCT network_region FROM nem_project.1_bronze.facilities ORDER BY network_region;

In [0]:
%sql
-- Check records withouth lat and long
select * from nem_project.1_bronze.facilities where lat is null or lng is null

In [0]:
%sql
-- check if every single record has "<p>" and "</p>".
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN facility_description LIKE '%<p>%' THEN 1 ELSE 0 END) AS has_open_p_tag,
  SUM(CASE WHEN facility_description LIKE '%</p>%' THEN 1 ELSE 0 END) AS has_close_p_tag
FROM nem_project.`1_bronze`.facilities

In [0]:
%sql
--Filter records without facility_description with "<p>" and "</p>"
select * from nem_project.`1_bronze`.facilities where facility_description NOT LIKE '%</p>%'

In [0]:
%sql
-- Distinct list of Fuel type:
SELECT DISTINCT fueltech_id FROM nem_project.1_bronze.facilities ORDER BY fueltech_id;

In [0]:
%sql
-- Status
SELECT DISTINCT status_id FROM nem_project.1_bronze.facilities ORDER BY status_id;

In [0]:
%sql
-- Dispatch type
select distinct(dispatch_type) from nem_project.1_bronze.facilities order by dispatch_type

In [0]:
%sql
-- Count the number of data missing.
SELECT
  SUM(CASE WHEN data_first_seen IS NULL THEN 1 ELSE 0 END) AS null_data_first_seen,
  SUM(CASE WHEN data_last_seen IS NULL THEN 1 ELSE 0 END) AS null_data_last_seen
FROM nem_project.`1_bronze`.facilities

#### Plots

In [0]:
#Plot Registered_capacity
df = spark.sql("""
    SELECT CAST(capacity_registered AS DOUBLE) AS capacity_registered
    FROM nem_project.1_bronze.facilities
    WHERE capacity_registered IS NOT NULL AND capacity_registered != 'null'
""").toPandas()


plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

mean_val = df['capacity_registered'].mean()
median_val = df['capacity_registered'].median()

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df['capacity_registered'], bins=30, color='#4C72B0', edgecolor='white', alpha=0.9)
ax.axvline(mean_val, color='#C44E52', linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:,.1f} MW')
ax.axvline(median_val, color='#55A868', linestyle='--', linewidth=1.5, label=f'Median: {median_val:,.1f} MW')

ax.set_title('Most NEM Facilities Have Small Registered Capacity')
ax.set_xlabel('Registered Capacity (MW)')
ax.set_ylabel('Number of Facilities')
ax.legend(frameon=True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('capacity_registered_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Facility Power Emissions

In [0]:
%sql
-- Schema and types
DESCRIBE nem_project.1_bronze.facility_power_emissions;

In [0]:
%sql
-- Sample rows
select * from nem_project.`1_bronze`.facility_power_emissions limit 20

In [0]:
%sql
-- Row count and basic shape
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT facility_code) AS distinct_facilities,
       COUNT(DISTINCT unit_code) AS distinct_units,
       MIN(time) AS earliest_time,
       MAX(time) AS latest_time
FROM nem_project.1_bronze.facility_power_emissions;

In [0]:
%sql
-- Check duplicate records
SELECT facility_code, unit_code, time, COUNT(*) AS n
FROM nem_project.1_bronze.facility_power_emissions
GROUP BY facility_code, unit_code, time
HAVING COUNT(*) > 1
LIMIT 20;

In [0]:
%sql
-- Count Null and missingness records per column.
SELECT
  SUM(CASE WHEN power IS NULL OR power = '' THEN 1 ELSE 0 END) AS missing_power_column,
  SUM(CASE WHEN emissions IS NULL OR emissions = '' THEN 1 ELSE 0 END) AS missing_emissions_column,
  SUM(CASE WHEN time IS NULL OR time = '' THEN 1 ELSE 0 END) AS missing_time_column
FROM nem_project.1_bronze.facility_power_emissions;

#### Plots

In [0]:
"Power Plot"

df = spark.sql("SELECT * FROM nem_project.`1_bronze`.facility_power_emissions")

random_units = (
    df.select("unit_code")
      .distinct()
      .orderBy(F.rand())
      .limit(10)
      .toPandas()["unit_code"]
      .tolist()
)

plot_df = df.filter(df.unit_code.isin(random_units)).toPandas()
plot_df["time"] = pd.to_datetime(plot_df["time"], utc=True)
plot_df["power"] = plot_df["power"].astype(float)
plot_df["is_missing"] = plot_df["power"].isna()
plot_df = plot_df.sort_values("time")

LINE_COLOR = "#4C72B0"  # seaborn default blue
MISSING_COLOR = "#999999"

units = sorted(plot_df["unit_code"].unique())

with plt.style.context({
    "font.family": "sans-serif",
    "axes.titlesize": 11,
    "axes.titleweight": "normal",
    "axes.titlelocation": "left",
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "grid.alpha": 0.3,
}):
    fig, axes = plt.subplots(5, 2, figsize=(13, 15), sharex=True, layout="constrained")
    axes = axes.flatten()

    for ax, unit in zip(axes, units):
        group = plot_df[plot_df["unit_code"] == unit]
        ax.plot(group["time"], group["power"], color=LINE_COLOR, linewidth=0.8)

        missing = group[group["is_missing"]]
        if not missing.empty:
            ax.scatter(missing["time"], [0] * len(missing), marker="x",
                       color=MISSING_COLOR, s=8, zorder=3)

        ax.set_title(unit)
        ax.set_ylabel("Power (MW)")
        ax.grid(True, linewidth=0.4)
        ax.margins(x=0.01)
        ax.tick_params(labelbottom=True)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        for label in ax.get_xticklabels():
            label.set_rotation(30)
            label.set_ha("right")

    fig.suptitle(
        "Hourly power output, 10 randomly sampled units (Sep 2025 \u2013 Aug 2026)\n"
        "Gray \u00d7 marks missing readings (not zero)",
        fontsize=12, fontweight="bold",
    )

fig.savefig("power_grid.png", dpi=300)
fig.savefig("power_grid.pdf")
plt.show()

In [0]:
"Emissions plot"

df = spark.sql("SELECT * FROM nem_project.`1_bronze`.facility_power_emissions")

random_units = (
    df.select("unit_code")
      .distinct()
      .orderBy(F.rand())
      .limit(10)
      .toPandas()["unit_code"]
      .tolist()
)

plot_df = df.filter(df.unit_code.isin(random_units)).toPandas()
plot_df["time"] = pd.to_datetime(plot_df["time"], utc=True)
plot_df["emissions"] = plot_df["emissions"].astype(float)
plot_df["is_missing"] = plot_df["emissions"].isna()
plot_df = plot_df.sort_values("time")

LINE_COLOR = "#DD8452"
MISSING_COLOR = "#999999"

units = sorted(plot_df["unit_code"].unique())

with plt.style.context({
    "font.family": "sans-serif",
    "axes.titlesize": 11,
    "axes.titleweight": "normal",
    "axes.titlelocation": "left",
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "grid.alpha": 0.3,
}):
    fig, axes = plt.subplots(5, 2, figsize=(13, 15), sharex=True, layout="constrained")
    axes = axes.flatten()

    for ax, unit in zip(axes, units):
        group = plot_df[plot_df["unit_code"] == unit]
        ax.plot(group["time"], group["emissions"], color=LINE_COLOR, linewidth=0.8)

        missing = group[group["is_missing"]]
        if not missing.empty:
            ax.scatter(missing["time"], [0] * len(missing), marker="x",
                       color=MISSING_COLOR, s=8, zorder=3)

        ax.set_title(unit)
        ax.set_ylabel("Emissions (t CO\u2082e)")
        ax.grid(True, linewidth=0.4)
        ax.margins(x=0.01)
        ax.tick_params(labelbottom=True)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        for label in ax.get_xticklabels():
            label.set_rotation(30)
            label.set_ha("right")

    fig.suptitle(
        "Hourly emissions, 10 randomly sampled units (Sep 2025 \u2013 Aug 2026)\n"
        "Gray \u00d7 marks missing readings (not zero)",
        fontsize=12, fontweight="bold",
    )

fig.savefig("emissions_grid.png", dpi=300)
fig.savefig("emissions_grid.pdf")
plt.show()

## 3. Region price demand

In [0]:
%sql
-- Describe
DESCRIBE nem_project.1_bronze.region_price_demand;

In [0]:
%sql
-- Sample rows
select * from nem_project.`1_bronze`.region_price_demand limit 20

In [0]:
%sql
-- Basic shape
SELECT COUNT(*) AS total_rows,
    COUNT(DISTINCT network_region) AS distinct_network_regions,
    MIN(time) AS earliest_time,
    MAX(time) AS latest_time
FROM nem_project.1_bronze.region_price_demand;

In [0]:
%sql
-- Missing records
SELECT
    SUM(CASE WHEN time IS NULL OR time = '' THEN 1 ELSE 0 END) AS missing_time_column,
	SUM(CASE WHEN price IS NULL OR price = '' THEN 1 ELSE 0 END) AS missing_price_column,
	SUM(CASE WHEN demand IS NULL OR demand = '' THEN 1 ELSE 0 END) AS missing_demand_column
FROM nem_project.1_bronze.region_price_demand;

In [0]:
%sql
-- Check duplicate records
SELECT facility_code, unit_code, time, COUNT(*) AS n
FROM nem_project.1_bronze.facility_power_emissions
GROUP BY facility_code, unit_code, time
HAVING COUNT(*) > 1
LIMIT 20;

## Plots

In [0]:
#Price vs Demand per State
df = spark.sql("""
    SELECT
        network_region,
        CAST(date_trunc('day', CAST(time AS TIMESTAMP)) AS DATE) AS date,
        AVG(CAST(NULLIF(price, 'null') AS DOUBLE)) AS avg_price,
        AVG(CAST(NULLIF(demand, 'null') AS DOUBLE)) AS avg_demand
    FROM nem_project.1_bronze.region_price_demand
    WHERE price IS NOT NULL AND price != 'null'
      AND demand IS NOT NULL AND demand != 'null'
    GROUP BY network_region, date_trunc('day', CAST(time AS TIMESTAMP))
    ORDER BY network_region, date
""").toPandas()

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
})

regions = sorted(df['network_region'].unique())
n_regions = len(regions)

fig, axes = plt.subplots(n_regions, 1, figsize=(12, 3.2 * n_regions), sharex=True)
if n_regions == 1:
    axes = [axes]

demand_color = '#4C72B0'
price_color = '#C44E52'

for ax, region in zip(axes, regions):
    subset = df[df['network_region'] == region]

    ax.plot(subset['date'], subset['avg_demand'], color=demand_color,
             linewidth=1.6, linestyle='-', label='Avg Demand (MW)')
    ax.set_ylabel('Demand (MW)', color=demand_color)
    ax.tick_params(axis='y', labelcolor=demand_color)
    ax.spines['top'].set_visible(False)

    ax2 = ax.twinx()
    ax2.plot(subset['date'], subset['avg_price'], color=price_color,
              linewidth=1.2, linestyle='--', alpha=0.85, label='Avg Price ($/MWh)')
    ax2.set_ylabel('Price ($/MWh)', color=price_color)
    ax2.tick_params(axis='y', labelcolor=price_color)
    ax2.spines['top'].set_visible(False)
    ax2.grid(False)  # prevent the twin axis's own gridlines competing with the left axis's

    ax.set_title(region)

# Build the legend from fixed handles, no extra twinx() calls
demand_handle = mlines.Line2D([], [], color=demand_color, linewidth=1.6, linestyle='-', label='Avg Demand (MW)')
price_handle = mlines.Line2D([], [], color=price_color, linewidth=1.2, linestyle='--', label='Avg Price ($/MWh)')

fig.suptitle('Daily Average Price and Demand by NEM Region (Sep 2025–Aug 2026)',
             fontsize=15, fontweight='bold', y=1.005)
fig.legend(handles=[demand_handle, price_handle], loc='upper center', ncol=2,
           bbox_to_anchor=(0.5, 0.995), frameon=True)

axes[-1].set_xlabel('Date')
fig.autofmt_xdate()
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('price_demand_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Changes to do in the silver layer

#### Facilities
**Changes to do**
* `network_id`: Drop the column.
* `network_region`: Remove the trailing "1" suffix from each value.
* `is_location_missing`: Create a flag indicating whether latitude or longitude is null.
* `facility_description`: Strip HTML tags (`<p>`, `</p>`).
* `fueltech_id`: Clean and standardise values.
* `fueltech_category`: Derive a cleaner category from `fueltech_id`.
* `dispatch_type`: Convert to lowercase.
* `data_is_missing`: Create a flag indicating whether `data_first_seen` or `data_last_seen` is null.

**Column Datatypes:**
* facility_code: STRING
* facility_name: STRING
* network_id: STRING
* network_region: STRING
* lat: DOUBLE
* lng: DOUBLE
* facility_description: STRING
* unit_code: STRING
* fueltech_id: STRING
* fueltech_category: STRING
* status_id: STRING
* capacity_registered: DOUBLE
* capacity_maximum: DOUBLE
* capacity_storage: DOUBLE
* data_first_seen: TIMESTAMP
* data_last_seen: TIMESTAMP
* dispatch_type: STRING
* is_location_missing: BOOLEAN
* data_is_missing: BOOLEAN

#### Facility Power Emissions
**Changes to do**
* `is_power_missing`: Create a flag indicating whether power is null.
* `is_emissions_missing`: Create a flag indicating whether emissions is null.

**Column Datatypes:**
* facility_code: STRING
* unit_code: STRING
* time: TIMESTAMP
* power: DOUBLE
* emissions: DOUBLE
* is_power_missing: BOOLEAN
* is_emissions_missing: BOOLEAN

#### Region Price demand
**Changes to do**
* `network_region`: Remove the trailing "1" suffix from each value.

**Column Datatypes:**
* network_region: STRING
* time: TIMESTAMP
* price: DOUBLE
* demand: DOUBLE